# ML-04 — Search Intelligence Data Contract

**Lane:** Content Performance Prediction / Refresh Opportunity Scoring

This contract uses the full FlyRank warehouse release. I use **March 2026** as the mid-panel feature month and **April 2026** as the forward outcome month. The goal is to rank content pages for review using signals that were available before the outcome window.

> The warehouse is gated. This notebook expects an `HF_TOKEN` Colab Secret or a secure token prompt. Never paste the token into a public cell.

In [ ]:
## 0. Connect to the warehouse

The daily fact is the source for time-series features and future outcomes. DuckDB reads the Parquet data remotely so the notebook does not load the full ~79M-row table into pandas.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-31"
OUTCOME_START = "2026-04-01"
OUTCOME_END = "2026-04-30"

print("Feature window:", FEATURE_START, "to", FEATURE_END)
print("Outcome window:", OUTCOME_START, "to", OUTCOME_END)

## 1. Unit of analysis + time window

**One row in my analysis = one pseudonymized content item for the March 2026 decision window.**

The underlying warehouse fact is daily: one row represents one `report_date × client_hash_id × content_hash_id`. I aggregate those daily rows into one March feature row per content item. The model/ranking decision is therefore made at the content-page level at the end of March.

**Feature window:** 2026-03-01 through 2026-03-31.

**Outcome window:** 2026-04-01 through 2026-04-30.

**Decision moment:** after March data is available and before April outcomes are known.

**Label/proxy:** `is_declining_next_30d = 1` when April impressions are less than 80% of March impressions, i.e. more than a 20% decline. This is a future observed outcome, not a product decision flag.

## 2. Fields: feature / label / context / excluded

### Features — safe at the March decision moment
1. `march_impressions` — total GSC impressions during March.
2. `march_clicks` — total GSC clicks during March.
3. `march_avg_position` — average observed GSC position during March.
4. `march_days_with_impressions` — number of March days with at least one impression.
5. `content_age_days_at_decision` — content age measured at 2026-03-31.

### Label
- `is_declining_next_30d` — 1 when April impressions are below 80% of March impressions.

### Context
- `client_hash_id` — grouping/splitting only.
- `content_hash_id` — page identity and joins only.
- `report_date` — used to define windows, not a model feature.

### Excluded
- April impressions/clicks/position as model features — they are future information relative to the March decision.
- `trend_pct` / `trend_direction` from a current or overlapping window — they can contain the outcome period and therefore create leakage.
- `ga4_*` fields for this first contract — the lane can proceed using GSC signals, and early history can have `ga4_data_available = FALSE`.
- Product decision flags/scores such as refresh or priority flags — they would copy an existing business rule rather than discover signal.
- Raw/private client names, URLs, queries, and titles — not part of the released analysis.

## 3. Verify it with queries

The three query cells below are the required checks. They verify:

1. **Grain:** no duplicate daily fact rows for the stated source grain.
2. **Counts and window:** the March slice row count, distinct content items, and date span.
3. **Availability:** how many March rows remain when GSC availability is explicitly required with `IS TRUE`.

The notebook intentionally keeps this to **exactly three verification queries**.

In [ ]:
# VERIFICATION QUERY 1 — grain
# One warehouse row should be one report_date × client × content combination.
q1 = f"""
SELECT
    COUNT(*) AS duplicate_grain_groups
FROM (
    SELECT report_date, client_hash_id, content_hash_id
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
) d
"""

grain_check = con.sql(q1).df()
display(grain_check)
print("Expected result: duplicate_grain_groups = 0")

In [ ]:
# VERIFICATION QUERY 2 — March count and date span
q2 = f"""
SELECT
    COUNT(*) AS march_fact_rows,
    COUNT(DISTINCT content_hash_id) AS march_content_items,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {TABLES['daily']}
WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
"""

march_check = con.sql(q2).df()
display(march_check)

In [ ]:
# VERIFICATION QUERY 3 — availability
# IS TRUE is deliberate: missing/unknown availability must not silently pass.
q3 = f"""
SELECT
    COUNT(*) AS march_rows_before_filter,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        * 100.0 / COUNT(*) AS gsc_available_pct
FROM {TABLES['daily']}
WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
"""

availability_check = con.sql(q3).df()
display(availability_check)

In [ ]:
## 3b. Five-feature frame

I build the feature frame only from March data, so every feature is available at the decision moment. The April outcome is joined only afterward to create the label.

| Feature | Available when? |
|---|---|
| `march_impressions` | Known after the March reporting window closes because it is summed only from March GSC observations. |
| `march_clicks` | Known after March closes because it uses only March GSC click observations. |
| `march_avg_position` | Known after March closes because it uses only March observed positions. |
| `march_days_with_impressions` | Known after March closes because it counts only March dates with impressions. |
| `content_age_days_at_decision` | Known on 2026-03-31 because it is calculated from content creation date to the decision date. |

In [ ]:
# Build the five-feature March frame.
# This is intentionally a small content-level result brought into pandas.
features_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS march_avg_position,
        COUNT(DISTINCT report_date)
            FILTER (WHERE COALESCE(gsc_impressions, 0) > 0) AS march_days_with_impressions
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
),
content_context AS (
    SELECT
        content_hash_id,
        MIN(content_created_at) AS content_created_at
    FROM {TABLES['content']}
    GROUP BY 1
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    m.march_days_with_impressions,
    DATE_DIFF(
        'day',
        CAST(c.content_created_at AS DATE),
        DATE '{FEATURE_END}'
    ) AS content_age_days_at_decision
FROM march m
LEFT JOIN content_context c USING (content_hash_id)
WHERE m.march_impressions > 0
"""

features = con.sql(features_sql).df()
print(f"Feature rows: {len(features):,}")
display(features.head())
display(features.describe(include="all").T)

In [ ]:
## 3c. Add the forward label — after the feature frame is built

The label is based on April impressions compared with March impressions. April data is **not** included in the five model features.

In [ ]:
outcome_sql = f"""
WITH march AS (
    SELECT
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1
),
april AS (
    SELECT
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1
)
SELECT
    m.content_hash_id,
    m.march_impressions,
    COALESCE(a.april_impressions, 0) AS april_impressions,
    CASE
        WHEN COALESCE(a.april_impressions, 0) < 0.80 * m.march_impressions
        THEN 1 ELSE 0
    END AS is_declining_next_30d
FROM march m
LEFT JOIN april a USING (content_hash_id)
WHERE m.march_impressions > 0
"""

outcomes = con.sql(outcome_sql).df()

model_df = features.merge(
    outcomes[["content_hash_id", "april_impressions", "is_declining_next_30d"]],
    on="content_hash_id",
    how="inner"
)

print(f"Final feature/label rows: {len(model_df):,}")
print(f"Future-decline rate: {model_df['is_declining_next_30d'].mean():.3%}")
display(model_df.head())

In [ ]:
## 3d. Deliberate leakage trap

To prove the leakage risk, I intentionally add **one label-derived column**: `leak_future_decline`, which is simply a copy of the future label.

This column is not a legitimate feature. It is added only to demonstrate how a model can appear nearly perfect when the answer is accidentally present in the inputs. After the demonstration, it is removed and the honest feature set is kept.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

honest_features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_days_with_impressions",
    "content_age_days_at_decision",
]

leak_df = model_df.dropna(subset=honest_features).copy()

X_train, X_test, y_train, y_test = train_test_split(
    leak_df[honest_features],
    leak_df["is_declining_next_30d"],
    test_size=0.25,
    random_state=42,
    stratify=leak_df["is_declining_next_30d"],
)

honest_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)
honest_model.fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])

# Deliberate leak: the future label itself is exposed as a feature.
leak_df["leak_future_decline"] = leak_df["is_declining_next_30d"]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    leak_df[honest_features + ["leak_future_decline"]],
    leak_df["is_declining_next_30d"],
    test_size=0.25,
    random_state=42,
    stratify=leak_df["is_declining_next_30d"],
)

leak_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1
)
leak_model.fit(X_train_l, y_train_l)
leak_auc = roc_auc_score(
    y_test_l,
    leak_model.predict_proba(X_test_l)[:, 1]
)

print(f"Honest feature AUC: {honest_auc:.3f}")
print(f"Leaky feature AUC:  {leak_auc:.3f}")
print("Leak removed from final feature list:", "leak_future_decline" not in honest_features)

In [ ]:
### Leakage lesson

The leaky score should move very close to perfect because the model has been handed the answer. That result is **not evidence of a good model**; it is evidence that the feature definition is invalid.

I therefore remove `leak_future_decline` and keep only the five March features. The honest score is the number I would report for this experiment.

The same rule applies to any current-window `trend_pct`, `trend_direction`, April metrics, or product decision field that overlaps the future outcome.

## 4. Data limits

1. **Unbalanced history:** clients do not all have the same tracking history, so March coverage is not equally deep across clients.
2. **GSC-only early rows:** GA4 availability can be false in early history. This contract therefore uses GSC signals and explicitly checks `gsc_data_available IS TRUE`.
3. **The label is directional:** a >20% month-over-month impression decline is a defined proxy for review risk. It does not prove that a page needs a refresh or that refreshing it will recover traffic.
4. **No causal claim:** this analysis cannot tell whether an update caused a performance change.
5. **Window dependence:** March features predict an April outcome. Features from April or overlapping 90-day windows would break the decision-time boundary and create leakage.
6. **Selection bias:** requiring measurable GSC availability and March impressions means the resulting ranking applies to observable content, not every piece of content in the warehouse.

## 5. Self-check

- [x] Unit of analysis and feature/outcome windows are stated.
- [x] Feature, label, context, and excluded fields are separated.
- [x] Exactly three verification query cells are included.
- [x] Availability is checked explicitly with `IS TRUE`.
- [x] Five features have an “available when?” explanation.
- [x] One deliberate label-derived leakage feature is added, tested, and removed.
- [x] Limitations are stated with careful, decision-support language.
- [ ] Run **Runtime → Run all** in Colab after adding your Hugging Face Secret and confirm the three query outputs and leakage scores are visible.
- [ ] Commit this executed notebook as `work/notebooks/w03_data_contract.ipynb`.